## Notebook 06: Regionalization Effect Analysis

This notebook quantifies how regionalization of the background database affects social indicator scores. It compares LCA results computed with the base ecoinvent database against those computed with the regionalized version.

**What it does:**
1. Builds matched functional units from the base and regionalized ecoinvent databases.
2. Runs LCA for all social impact methods on both database versions.
3. Computes log10(regionalized / base) per functional unit and indicator.
4. Aggregates results per location (median across indicators, then across functional units).
5. Produces a boxplot showing the top and bottom locations by effect size.

**Inputs:**
- Brightway project with social flows from notebook 01.
- Base database (`ecoinvent-3.10.1-cutoff`) and regionalized database (`ecoinvent-3.10.1-cutoff regionalized`).

**Outputs:**
- `regionalization_effect.svg`: boxplot of regionalization effect by location (174 mm wide).

In [ ]:
# %% -------------------------------------------------------------------
# 0. Imports
# ----------------------------------------------------------------------
from brightway2 import *
from bw2calc import LCA

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from pathlib import Path
import os

# %% -------------------------------------------------------------------
# 1. USER CONFIGURATION (adjust here for reproduction)
# ----------------------------------------------------------------------
# Brightway project and databases
PROJECT_NAME = "sense_project"                     # Brightway project name

DB_BASE = "ecoinvent-3.10.1-cutoff"                # Base database
DB_REGIONALIZED = "ecoinvent-3.10.1-cutoff regionalized"  # Regionalized database

# Calculation setup names (used for caching in Brightway)
SETUP_NAME_BASE = "my_batch_run_base"
SETUP_NAME_REG = "my_batch_run_regionalized"

# Name of batch run (for human reference only)
CS_NAME = "my_batch_run"

# Amount of each functional unit (can be changed if needed)
GLOBAL_AMOUNT = 1.0

# Number of activities to process (set to None for all common activities)
NUM_ACTIVITIES_TO_PROCESS = None

# Output directory for CSV result files
OUTPUT_DIR = Path(r"../Data/case_study_results/hydrogen_case_study_results")

# Terms for excluding activities (markets, mixes, etc.)
EXCLUDE_TERMS_STARTING_WITH = [
    "market for",
    "market group for",
]
EXCLUDE_TERMS = [
    "generic market",
    "import from",
    "production mix",
    "consumption mix",
    "cut-off",
    "cut off",
    "cutoff",
    "technology mix",
    "consumption market for",
]

# Impact methods: only those with first element in TARGET_GROUPS will be used
TARGET_GROUPS = [
    "Local Community",
    "Society",
    "Value Chain Actors",
    "Workers",
    "Children",
    "Consumer",
]

# Plot configuration
PLOT_KWARGS = dict(
    topN=5,
    min_fus=30,
    extra_space=2.5,
    middle_spacing=0.3,
    width_mm=174.0,
    height_inches=5.0,
    label_style="shorten",
    label_maxlen=24,
    # optional:
    x_ticks=(-1.0, -0.5, 0.0, 0.5, 1.0),
    x_lim=(-0.75, 0.75),
    show_middle_bracket=True,
)

# %% -------------------------------------------------------------------
# 2. Brightway project setup and database utilities
# ----------------------------------------------------------------------
def set_project(project_name: str) -> None:
    """Set the active Brightway project."""
    projects.set_current(project_name)
    print(f"Active Brightway project: {projects.current}")


def build_by_triple(db: Database) -> dict:
    """
    Build an index mapping (name, reference product, location) to activity keys.
    This is used to match activities between the base and regionalized databases.
    """
    by_triple = {}
    for act in db:
        triple = (act.get("name"), act.get("reference product"), act.get("location"))
        by_triple.setdefault(triple, []).append(act.key)
    return by_triple


def should_exclude_activity(
    activity, 
    exclude_terms_starting_with=None, 
    exclude_terms=None
) -> bool:
    """
    Determine whether an activity should be excluded based on its name and
    reference product.
    """
    if exclude_terms_starting_with is None:
        exclude_terms_starting_with = EXCLUDE_TERMS_STARTING_WITH
    if exclude_terms is None:
        exclude_terms = EXCLUDE_TERMS

    name = activity.get("name", "")
    reference_product = activity.get("reference product", "")

    # Exclude if the name starts with any of the exclusion terms
    if any(name.lower().startswith(term.lower()) for term in exclude_terms_starting_with):
        return True

    # Exclude if the name or reference product contains any of the exclusion terms
    if any(
        term.lower() in name.lower() or term.lower() in reference_product.lower()
        for term in exclude_terms
    ):
        return True

    return False

# %% -------------------------------------------------------------------
# 3. Construct functional units and calculation setups
# ----------------------------------------------------------------------
def construct_functional_units(
    db_base_name: str,
    db_regionalized_name: str,
    global_amount: float = 1.0,
    max_activities: int | None = None,
) -> tuple[list[dict], list[dict]]:
    """
    Build lists of functional units for base and regionalized databases,
    matched by (name, reference product, location), and filtered by exclusion rules.

    Assumption (verified in your case):
      - (name, reference product, location) is UNIQUE within each database.
    If this assumption is violated, the function fails loudly.
    """
    db_base = Database(db_base_name)
    db_reg  = Database(db_regionalized_name)

    base_by_triple = build_by_triple(db_base)
    reg_by_triple  = build_by_triple(db_reg)

    common_triples = sorted(set(base_by_triple.keys()).intersection(reg_by_triple.keys()))
    print(f"Found {len(common_triples)} common activities (name, reference product, location).")

    if max_activities is not None:
        common_triples = common_triples[:max_activities]
        print(f"Limiting to first {len(common_triples)} common activities for processing.")

    func_units_base: list[dict] = []
    func_units_reg:  list[dict] = []

    for triple in common_triples:
        # ---- Base database activity (expect exactly 1 key per triple) ----
        keys_base = base_by_triple.get(triple, [])
        if len(keys_base) != 1:
            raise ValueError(
                f"Non-unique or missing triple in BASE DB for {triple}: "
                f"expected 1 activity, found {len(keys_base)}."
            )
        key_base = keys_base[0]
        activity_base = get_activity(key_base)
        if should_exclude_activity(activity_base):
            continue

        # ---- Regionalized database activity (expect exactly 1 key per triple) ----
        keys_reg = reg_by_triple.get(triple, [])
        if len(keys_reg) != 1:
            raise ValueError(
                f"Non-unique or missing triple in REGIONALIZED DB for {triple}: "
                f"expected 1 activity, found {len(keys_reg)}."
            )
        key_reg = keys_reg[0]
        activity_reg = get_activity(key_reg)
        if should_exclude_activity(activity_reg):
            continue

        # ---- Add matched functional units ----
        func_units_base.append({key_base: global_amount})
        func_units_reg.append({key_reg: global_amount})

    print(f"Functional units (base):         {len(func_units_base)}")
    print(f"Functional units (regionalized): {len(func_units_reg)}")

    # sanity check: should remain aligned one-to-one
    if len(func_units_base) != len(func_units_reg):
        raise RuntimeError(
            "Base and regionalized functional unit lists differ in length. "
            "This should not happen under one-to-one matching."
        )

    return func_units_base, func_units_reg


def select_impact_methods(target_groups: list[str]) -> list:
    """
    Select impact assessment methods whose first element is in target_groups.
    """
    selected = [m for m in methods if m[0] in target_groups]
    print(f"Selected {len(selected)} impact methods.")
    return selected


def save_calculation_setups(
    setup_name_base: str,
    setup_name_reg: str,
    func_units_base: list[dict],
    func_units_reg: list[dict],
    impact_methods: list,
) -> None:
    """Save Brightway calculation setups for base and regionalized FUs."""
    calculation_setups[setup_name_base] = {
        "inv": func_units_base,
        "ia": impact_methods,
    }
    calculation_setups[setup_name_reg] = {
        "inv": func_units_reg,
        "ia": impact_methods,
    }

    print(f"\nSetup saved as '{setup_name_base}' (base database).")
    print(f"Functional units (base):         {len(func_units_base)}")
    print(f"\nSetup saved as '{setup_name_reg}' (regionalized database).")
    print(f"Functional units (regionalized): {len(func_units_reg)}")

# %% -------------------------------------------------------------------
# 4. Run LCAs and export total scores to CSV
# ----------------------------------------------------------------------
def process_func_units_and_lca(
    cs: dict,
    setup_name: str,
    output_dir: Path,
    num_activities: int | None = None,
    overwrite_existing: bool = False,
) -> Path:
    """
    Run LCA for each functional unit in a calculation setup and
    export total scores (one row per activity, one column per method) to CSV.

    If the CSV already exists and overwrite_existing=False, the calculation is skipped.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    func_units = cs["inv"]
    methods_list = cs["ia"]
    method_labels = [" | ".join(m) for m in methods_list]

    if num_activities is not None:
        func_units = func_units[:num_activities]

    output_csv_path = output_dir / f"{setup_name}_total_scores.csv"

    if output_csv_path.exists() and not overwrite_existing:
        print(f"File for {setup_name} already exists at '{output_csv_path}'. Skipping LCA.")
        return output_csv_path

    # Initialize LCA with first functional unit and factorize LCI
    print(f"\nInitializing LCA for setup '{setup_name}' ...")
    lca = LCA(func_units[0])
    lca.lci(factorize=True)

    # Precompute characterization matrices for each method
    cf_matrices = {}
    for method in methods_list:
        lca.switch_method(method)
        cf_matrices[method] = lca.characterization_matrix.copy()

    results = []
    for fu in tqdm(func_units, desc=f"Processing FUs in {setup_name}"):
        row_score = {}
        fu_key = list(fu.keys())[0]

        # Recalculate LCI for the current functional unit
        lca.redo_lci(fu)

        # Activity metadata
        activity = get_activity(fu_key)
        row_score["name"] = activity["name"]
        row_score["reference product"] = activity["reference product"]
        row_score["location"] = activity["location"]
        row_score["unit"] = activity.get("unit", "")
        row_score["database"] = activity["database"]

        # LCA scores for each method
        for j, method in enumerate(methods_list):
            lca.characterization_matrix = cf_matrices[method]
            lca.lcia_calculation()
            row_score[method_labels[j]] = lca.score

        results.append(row_score)

    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv_path, index=False)

    print(
        f"\nTotal scores for {len(results)} functional units from '{setup_name}' "
        f"written to '{output_csv_path}'."
    )
    return output_csv_path

# %% -------------------------------------------------------------------
# 5. Match base and regionalized results
# ----------------------------------------------------------------------
def match_base_and_regionalized(
    base_csv: Path,
    reg_csv: Path,
    key_cols: list[str] = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Match base and regionalized results on key columns.

    Returns
    -------
    matched : DataFrame
        Inner join of base and regionalized results.
    only_in_base : DataFrame
        Activities only present in the base CSV.
    only_in_reg : DataFrame
        Activities only present in the regionalized CSV.
    """
    if key_cols is None:
        key_cols = ["name", "reference product", "location", "unit"]

    base_df = pd.read_csv(base_csv)
    reg_df = pd.read_csv(reg_csv)

    for col in key_cols:
        if col not in base_df.columns:
            raise ValueError(f"Column '{col}' missing in base_df")
        if col not in reg_df.columns:
            raise ValueError(f"Column '{col}' missing in reg_df")

    matched = pd.merge(
        base_df, reg_df,
        on=key_cols, how="inner",
        suffixes=("_base","_reg"),
        validate="one_to_one",
    )

    outer = pd.merge(
        base_df[key_cols],
        reg_df[key_cols],
        on=key_cols,
        how="outer",
        indicator=True,
    )

    only_in_base = outer[outer["_merge"] == "left_only"]
    only_in_reg = outer[outer["_merge"] == "right_only"]

    print("=== Matching summary ===")
    print(f"Base rows total:          {len(base_df)}")
    print(f"Regionalized rows total:  {len(reg_df)}")
    print(f"Matched activities:       {len(matched)}")
    print(f"Unmatched only in base:   {len(only_in_base)}")
    print(f"Unmatched only in region.:{len(only_in_reg)}")

    if not only_in_base.empty:
        print("\nExample activities only in BASE:")
        print(only_in_base.head())

    if not only_in_reg.empty:
        print("\nExample activities only in REGIONALIZED:")
        print(only_in_reg.head())

    return matched, only_in_base, only_in_reg

# %% -------------------------------------------------------------------
# 6. Plot
# ----------------------------------------------------------------------

def plot_regionalization_effect_by_location_fu_median(
    matched: pd.DataFrame,
    topN: int = 5,
    min_fus: int = 30,
    social_only: bool = True,
    stakeholders=None,
    focus_stakeholder: str | None = None,   # e.g. "Workers" or None
    extra_space: float = 2.5,
    middle_spacing: float = 0.12,
    width_mm: float = 174.0,
    height_inches: float = 5.0,
    label_style: str = "shorten",           # "shorten", "wrap", "full"
    label_maxlen: int = 24,
    # NEW: x-axis ticks in log10 units (numbers like -1, -0.5, 0, 0.5, 1)
    x_ticks=(-1.0, -0.5, 0.0, 0.5, 1.0),
    x_lim=None,  # e.g. (-0.8, 0.8) or None for auto
    # NEW: bracket label for middle locations on y-axis
    show_middle_bracket: bool = True,
    middle_label: str = "Other locations\n(median only)",
):
    """
    Dependence-aware location plot:
      1) Δ_ij = log10(reg/base) per FU i and indicator j
      2) Δ_i  = median_j(Δ_ij) per FU (accounts for correlated indicators)
      3) For each location: median of Δ_i across FUs

    Display:
      - middle locations as black points (medians only)
      - top/bottom N locations as horizontal boxplots
      - bracket + label on y-axis spanning the middle region
      - x-axis ticks shown as numeric log10 values (e.g., -1, -0.5, 0, 0.5, 1)
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    if stakeholders is None:
        stakeholders = ["workers", "society", "local community", "value chain actors", "consumer", "children"]

    if "location" not in matched.columns:
        raise ValueError("matched must contain a 'location' column.")

    # ---- Find *_base / *_reg indicator pairs ----
    pairs = []
    for col in matched.columns:
        if not col.endswith("_base"):
            continue
        ind = col[:-5]
        if social_only:
            first = ind.split(" | ")[0].strip().lower()
            if first not in stakeholders:
                continue
        reg_col = ind + "_reg"
        if reg_col in matched.columns:
            pairs.append((col, reg_col, ind))

    if not pairs:
        raise ValueError("No indicator pairs found to compute Δ.")

    # Optional: filter indicators by stakeholder
    if focus_stakeholder is not None:
        fs = focus_stakeholder.strip().lower()
        pairs = [p for p in pairs if p[2].split(" | ")[0].strip().lower() == fs]
        if not pairs:
            raise ValueError(f"No indicator pairs match focus_stakeholder={focus_stakeholder!r}")

    base_cols = [p[0] for p in pairs]
    reg_cols  = [p[1] for p in pairs]

    base = matched[base_cols].apply(pd.to_numeric, errors="coerce").to_numpy()
    reg  = matched[reg_cols].apply(pd.to_numeric, errors="coerce").to_numpy()

    with np.errstate(divide="ignore", invalid="ignore"):
        logdiff = np.log10(reg / base)

    valid = np.isfinite(logdiff) & (base > 0) & (reg > 0)
    logdiff = np.where(valid, logdiff, np.nan)

    # ---- FU-level aggregation across indicators (dependence-aware) ----
    fu_delta = np.nanmedian(logdiff, axis=1)
    valid_fu = np.isfinite(fu_delta)

    fu_delta = fu_delta[valid_fu]
    locs = matched.loc[valid_fu, "location"].astype(str).to_numpy()

    # ---- Pool FU-level values per location ----
    pooled = {}
    for loc in pd.unique(locs):
        vals = fu_delta[locs == loc]
        vals = vals[np.isfinite(vals)]
        if vals.size >= min_fus:
            pooled[loc] = vals

    if not pooled:
        raise ValueError(f"No locations pass min_fus={min_fus}. Lower min_fus or check data validity.")

    loc_meds = {loc: float(np.median(vals)) for loc, vals in pooled.items()}
    loc_ns   = {loc: int(vals.size) for loc, vals in pooled.items()}

    summary_df = (
        pd.DataFrame({
            "location": list(loc_meds.keys()),
            "median_fu": list(loc_meds.values()),
            "n_fus": [loc_ns[l] for l in loc_meds.keys()],
        })
        .sort_values("median_fu", ascending=True)
        .reset_index(drop=True)
    )

    # ---- split into bottom/middle/top ----
    sorted_locs = summary_df["location"].tolist()
    topN = min(topN, max(1, len(sorted_locs) // 2))
    bottom_locs = sorted_locs[:topN]
    top_locs = sorted_locs[-topN:]
    middle_locs = sorted_locs[topN:-topN]

    # ---- y positioning ----
    y_positions = {}
    y = 0.0
    for i, loc in enumerate(bottom_locs):
        y_positions[loc] = y
        if i < len(bottom_locs) - 1:
            y += extra_space
    if bottom_locs and middle_locs:
        y += extra_space / 2
    for j, loc in enumerate(middle_locs):
        y_positions[loc] = y
        if j < len(middle_locs) - 1:
            y += middle_spacing
    if middle_locs and top_locs:
        y += extra_space / 2
    for k, loc in enumerate(top_locs):
        y_positions[loc] = y
        if k < len(top_locs) - 1:
            y += extra_space

    # ---- label helpers ----
    def shorten(s, maxlen=24):
        s = str(s)
        return s if len(s) <= maxlen else s[:maxlen - 1] + "…"

    def fmt_label(loc):
        lab = loc
        if label_style == "shorten":
            lab = shorten(lab, label_maxlen)
        return f"{lab} (n={loc_ns[loc]})"

    # ---- plot ----
    fig_w = width_mm / 25.4
    fig, ax = plt.subplots(figsize=(fig_w, height_inches))

    # middle points (medians only)
    meds_mid = None
    ys_mid = None
    if middle_locs:
        meds_mid = summary_df.set_index("location").loc[middle_locs, "median_fu"].to_numpy()
        ys_mid = np.array([y_positions[l] for l in middle_locs], dtype=float)
        ax.scatter(meds_mid, ys_mid, s=14, color="black", zorder=3)

    # highlighted boxplots
    hi_locs = bottom_locs + top_locs
    hi_data = [pooled[l] for l in hi_locs]
    hi_y = [y_positions[l] for l in hi_locs]

    ax.boxplot(
        hi_data,
        positions=hi_y,
        vert=False,
        widths=2.0,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(facecolor="lightblue", alpha=0.6),
        medianprops=dict(color="red", linewidth=1.5),
        whiskerprops=dict(color="gray", linewidth=1),
        capprops=dict(color="gray", linewidth=1),
    )

    # y labels for highlighted only
    ax.set_yticks([y_positions[l] for l in hi_locs])
    ax.set_yticklabels([fmt_label(l) for l in hi_locs])

    # x-axis: numeric log10 ticks
    ax.set_xticks(list(x_ticks))
    ax.set_xticklabels([f"{t:g}" for t in x_ticks])

    # reference line at 0 only (since you prefer numeric ticks)
    ax.axvline(0, color="grey", linestyle="--", linewidth=1)

    # x limits
    if x_lim is not None:
        ax.set_xlim(*x_lim)
    else:
        # auto but keep a bit of padding
        all_vals = np.concatenate(
            [summary_df["median_fu"].to_numpy()] + [np.asarray(v, dtype=float) for v in pooled.values()]
        )
        all_vals = all_vals[np.isfinite(all_vals)]
        if all_vals.size:
            lo, hi = np.quantile(all_vals, [0.01, 0.99])
            pad = 0.08 * max(1e-6, (hi - lo))
            ax.set_xlim(lo - pad, hi + pad)

    # axis label (clear + numeric ticks)
    ax.set_xlabel("Change in social indicator scores due to regionalization\n"
                  r"$\Delta=\log_{10}(\mathrm{regionalized}/\mathrm{base})$  (right = higher scores)")

    # ---- Middle bracket + y-axis label ----
    if show_middle_bracket and (middle_locs and ys_mid is not None and ys_mid.size > 1):
        y_low = float(np.min(ys_mid))
        y_high = float(np.max(ys_mid))
        y_mid = 0.5 * (y_low + y_high)

        x0, x1 = ax.get_xlim()
        xr = x1 - x0
        x_br = x0 - 0.06 * xr  # slightly left of plot area
        cap = 0.02 * xr        # bracket "cap" length

        # bracket line and caps (drawn in data coords)
        ax.plot([x_br, x_br], [y_low, y_high], color="black", linewidth=1, clip_on=False)
        ax.plot([x_br, x_br + cap], [y_low, y_low], color="black", linewidth=1, clip_on=False)
        ax.plot([x_br, x_br + cap], [y_high, y_high], color="black", linewidth=1, clip_on=False)

        # label centered on bracket
        ax.text(
            x_br - 0.01 * xr, y_mid,
            middle_label,
            ha="right", va="center",
            fontsize=9,
            clip_on=False
        )

    ys = list(y_positions.values())
    ax.set_ylim(min(ys) - 1.5, max(ys) + 1.5)

    fig.tight_layout()
    return fig, ax, summary_df

# %% -------------------------------------------------------------------
# 7. Full pipeline execution (one cell to reproduce the figure)
# ----------------------------------------------------------------------
set_project(PROJECT_NAME)

# 1) Build functional units from base and regionalized databases
func_units_base, func_units_reg = construct_functional_units(
    db_base_name=DB_BASE,
    db_regionalized_name=DB_REGIONALIZED,
    global_amount=GLOBAL_AMOUNT,
    max_activities=NUM_ACTIVITIES_TO_PROCESS,
)

# 2) Select impact methods
impact_methods = select_impact_methods(TARGET_GROUPS)

# 3) Save calculation setups
save_calculation_setups(
    setup_name_base=SETUP_NAME_BASE,
    setup_name_reg=SETUP_NAME_REG,
    func_units_base=func_units_base,
    func_units_reg=func_units_reg,
    impact_methods=impact_methods,
)

cs_base = calculation_setups[SETUP_NAME_BASE]
cs_reg = calculation_setups[SETUP_NAME_REG]

# 4) Run LCAs (skips if CSVs already exist, unless overwrite_existing=True)
base_csv = process_func_units_and_lca(
    cs=cs_base,
    setup_name=SETUP_NAME_BASE,
    output_dir=OUTPUT_DIR,
    num_activities=NUM_ACTIVITIES_TO_PROCESS,
    overwrite_existing=False,
)
reg_csv = process_func_units_and_lca(
    cs=cs_reg,
    setup_name=SETUP_NAME_REG,
    output_dir=OUTPUT_DIR,
    num_activities=NUM_ACTIVITIES_TO_PROCESS,
    overwrite_existing=False,
)

# 5) Match base and regionalized results
matched, only_in_base, only_in_reg = match_base_and_regionalized(
    base_csv=base_csv,
    reg_csv=reg_csv,
)

# 6) Plot the effect of regionalization on social indicators
fig, ax, loc_summary = plot_regionalization_effect_by_location_fu_median(
    matched=matched,
    **PLOT_KWARGS,
)

FIG_SAVE_PATH = Path("regionalization_effect.svg")
FIG_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_SAVE_PATH, bbox_inches="tight")
print(f"Saved figure to: {FIG_SAVE_PATH.resolve()}")
plt.show()